In [ ]:
import numpy as np

# 1. 空间网格离散化
N = 100  # 网格区间数
h = 1.0 / N  # 空间步长
x = np.linspace(0, 1, N + 1)  # 全区间节点
print(x)
x_internal = x[1:-1]  # 内部待求节点 (N-1 个)
n_inner = N - 1

# 2. 使用 np.diag 构造三对角矩阵 A
main_diag = np.diag(2.0 * np.ones(n_inner), k=0)
print(main_diag)
upper_diag = np.diag(-1.0 * np.ones(n_inner - 1), k=1)
print(upper_diag)
lower_diag = np.diag(-1.0 * np.ones(n_inner - 1), k=-1)
print(lower_diag)
A = main_diag + upper_diag + lower_diag

# 3. 构造右端项向量 b
f = -(np.pi**2) * np.sin(np.pi * x_internal)
b = -(h**2) * f

# 4. 求解线性方程组 A * u_inner = b
u_inner = np.linalg.solve(A, b)
print(u_inner)

# 5. 拼接边界条件 u(0)=0, u(1)=0
u_numerical = np.concatenate(([0.0], u_inner, [0.0]))
u_exact = np.sin(np.pi * x)
print(u_exact)

# 6. 验证最大绝对误差
max_error = np.max(np.abs(u_numerical - u_exact))
print(f"网格数 N = {N}")
print(f"最大绝对误差 L_inf: {max_error:.2e}")

In [ ]:
import numpy as np


def apply_rotary_emb_complex(x: np.ndarray, seq_len: int, head_dim: int, base: float = 10000.0) -> np.ndarray:
    """
    参数说明:
    x: 输入的 Query 或 Key 向量，形状为 (seq_len, head_dim)，dtype 为 float32
    head_dim: 必须为偶数
    """
    # 1. 计算每个维度对的基础旋转频率 theta_i = base^(-2i / head_dim)
    # dim_indices: [0, 2, 4, ..., head_dim-2]
    dim_indices = np.arange(0, head_dim, 2, dtype=np.float32)

    print(dim_indices)
    inv_freq = 1.0 / (base ** (dim_indices / head_dim))  # 形状: (head_dim // 2,)
    print(inv_freq)

    # 2. 生成所有 token 位置 m: [0, 1, 2, ..., seq_len - 1]
    positions = np.arange(seq_len, dtype=np.float32)  # 形状: (seq_len,)
    print(positions)

    # 3. 外积生成每个位置对应的旋转相位 m * theta: 形状为 (seq_len, head_dim // 2)
    angles = np.outer(positions, inv_freq)
    print(angles)

    # 4. 构造复数旋转因子: exp(i * angles) = cos(angles) + i * sin(angles)
    # 这一步直接生成 complex64 数组
    rotary_factors = np.exp(1j * angles).astype(np.complex64)
    print(rotary_factors)

    # 5. 零拷贝: 将输入 x (seq_len, head_dim) 的 float32 视为 complex64
    # (x_0, x_1) 自动折叠为一个复数 x_0 + i * x_1，形状变为 (seq_len, head_dim // 2)
    x_complex = x.astype(np.float32).view(np.complex64)

    # 6. 一步完成旋转: 复数元素乘法 (广播机制)
    x_rotated_complex = x_complex * rotary_factors

    # 7. 零拷贝还原回实数 float32 格式，恢复形状 (seq_len, head_dim)
    x_rotated = x_rotated_complex.view(np.float32)

    return x_rotated


# ----------------- 测试示例 -----------------
seq_len = 4  # 4 个 Token
head_dim = 8  # 每个 Head 的维度为 8 (对应 4 个复数)

# 随机生成一个 Query 向量
np.random.seed(42)
q = np.random.randn(seq_len, head_dim).astype(np.float32)

q_out = apply_rotary_emb_complex(q, seq_len, head_dim)

print("原始 Q 形状:", q.shape)
print("旋转后 Q 形状:", q_out.shape)
print("\nToken 0 原始值前两维:\n", q[0, :2])
print("Token 0 旋转后前两维 (位置 0 不旋转，应相等):\n", q_out[0, :2])
print("\nToken 1 原始值前两维:\n", q[1, :2])
print("Token 1 旋转后前两维 (发生相位偏转):\n", q_out[1, :2])

In [ ]:
np.arange(0, 10, 2)

In [ ]:
np.logspace(2.0, 3.0, 4)